In [3]:
%pip install -q "pandas>=2.3,<3.0" anndata scanpy
import numpy as np
import pandas as pd

In [4]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_parquet('/content/drive/MyDrive/Datasets/fydp/tcga_brca_tri_modal_dataset.parquet')  # adjust folder path if inside a subfolder
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,vital_status,event,survival_time,age_at_index,ajcc_pathologic_stage,ER_status,PR_status,HER2_status,days_to_death,days_to_last_follow_up,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,Alive,0.0,4047.0,55.0,Stage X,Positive,Positive,Negative,NaN,4047.0,...,0.0,0.0000,0.5171,0.0,0.0000,0.0000,30.172701,0.0,0.1323,1.4353
TCGA-3C-AALI,Alive,0.0,4005.0,50.0,Stage IIB,Positive,Positive,Positive,NaN,4005.0,...,0.0,1.5146,0.7494,0.0,0.0000,0.0228,13.930200,0.0,0.0850,2.4341
TCGA-3C-AALJ,Alive,0.0,1474.0,62.0,Stage IIB,Positive,Positive,Equivocal,NaN,1474.0,...,0.0,2.3066,0.8500,0.0,0.0000,0.0000,11.335200,0.0,0.0719,2.2242
TCGA-3C-AALK,Alive,0.0,1448.0,52.0,Stage IA,Positive,Positive,Positive,NaN,1448.0,...,0.0,0.0000,0.5027,0.0,1.0673,0.0000,9.980100,0.0,0.0947,2.0927
TCGA-4H-AAAK,Alive,0.0,348.0,50.0,Stage IIIA,Positive,Positive,Equivocal,NaN,348.0,...,0.0,0.0000,0.5134,0.0,0.0000,0.0000,11.657800,0.0,0.0521,1.7913


In [5]:
df.shape

(1073, 60771)

In [6]:
df.isna().sum()

,0
vital_status,0
event,0
survival_time,0
age_at_index,0
ajcc_pathologic_stage,11
...,...
ENSG00000288669.1,0
ENSG00000288670.1,0
ENSG00000288671.1,0
ENSG00000288674.1,0


In [7]:
cut_idx = df.columns.get_loc('MUT_ANKRD12') + 1
df_rna = df.iloc[:, cut_idx:]
df_rna.shape

(1073, 60660)

In [8]:
print('Total missing values:', df_rna.isna().sum().sum())
print('Columns with missing values:', (df_rna.isna().sum() > 0).sum())
print('Duplicate rows:', df_rna.duplicated().sum())
print('Duplicate patient IDs:', df_rna.index.duplicated().sum())

Total missing values: 0
Columns with missing values: 0
Duplicate rows: 0
Duplicate patient IDs: 0


In [9]:
from huggingface_hub import login
from transformers import AutoModel, AutoTokenizer

login()

model_name = 'ScientaLab/eva-rna'
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True, device_map='auto')
model.eval()
print('Model loaded on:', next(model.parameters()).device)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

configuration_eva_rna.py:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ScientaLab/eva-rna:
- configuration_eva_rna.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

tokenization_eva_rna.py:   0%|          | 0.00/9.10k [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  194MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [10]:
import anndata as ad
import scanpy as sc

# 1. Strip Ensembl version suffix
base_genes = df_rna.columns.str.split(".").str[0]
df_rna_unver = df_rna.copy()
df_rna_unver.columns = base_genes
if df_rna_unver.columns.duplicated().any():
    df_rna_unver = df_rna_unver.T.groupby(level=0).sum().T

# 2. Keep only genes in eva-rna tokenizer vocabulary
ensembl_mapper = tokenizer.gene_mapper["ensembl_to_ncbi"]
mapped_genes = [g for g in df_rna_unver.columns if g in ensembl_mapper]
print(f"Total unversioned genes: {df_rna_unver.shape[1]} | Matched in eva-rna: {len(mapped_genes)}")
df_rna_eva = df_rna_unver[mapped_genes]

# 3. Build AnnData
adata = ad.AnnData(
    X=df_rna_eva.to_numpy(dtype="float32"),
    obs=pd.DataFrame(index=df_rna_eva.index),
    var=pd.DataFrame(index=df_rna_eva.columns),
)

# 4. Normalise to 1e6 (data is already TPM but scanpy HVG needs raw-ish counts)
#    Log1p first so HVG selection uses seurat_v3 (expects raw counts — use flavor="cell_ranger" instead)
sc.pp.log1p(adata, base=2)

# 5. Select top 2000 highly variable genes (reduces seq length from 26525 -> 2000)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="cell_ranger")
adata = adata[:, adata.var.highly_variable].copy()
print(f"AnnData after HVG selection: {adata.shape}")


Total unversioned genes: 60616 | Matched in eva-rna: 26525
AnnData after HVG selection: (1073, 2000)


In [11]:
import torch, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

with torch.no_grad():
    embeddings = model.encode_anndata(
        tokenizer, adata,
        species="human",
        batch_size=16,
    )

embeddings = embeddings.astype("float32")
print("Embeddings shape:", embeddings.shape)

emb_cols = [f"eva_{i}" for i in range(embeddings.shape[1])]
df_eva_emb = pd.DataFrame(embeddings, index=adata.obs_names, columns=emb_cols)
df_eva_emb.to_parquet("rna_eva_embeddings.parquet")
df_eva_emb


/root/.cache/huggingface/modules/transformers_modules/ScientaLab/eva_hyphen_rna/4232edf3c7efb01e4bb828be016f3fc86600f3e6/modeling_eva_rna.py:330: UserWarning: Auto-converted 2000/2000 genes from ensembl to NCBI GeneIDs (species: human). Set species='human'/'mouse' explicitly if auto-detection is wrong.
  return encode_from_anndata(
Encoding samples: 100%|██████████| 68/68 [04:02<00:00,  3.56s/batch]

Embeddings shape: (1073, 256)


,eva_0,eva_1,eva_2,eva_3,eva_4,eva_5,eva_6,eva_7,eva_8,eva_9,...,eva_246,eva_247,eva_248,eva_249,eva_250,eva_251,eva_252,eva_253,eva_254,eva_255
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,-2.155622,-0.291868,0.169682,-3.533919,-0.281709,-2.095984,1.502106,1.295753,0.824450,1.147492,...,0.673566,-2.987129,0.246001,-1.612530,-0.914301,-0.229371,1.868367,-0.552698,0.517181,-2.520375
TCGA-3C-AALI,-2.378538,-0.091101,0.088475,-3.407782,-0.398164,-2.097007,1.655441,0.476619,1.194781,1.192048,...,0.011487,-2.996671,-0.277262,-1.663273,-0.839044,-0.232655,1.803326,0.155092,0.614532,-2.153844
TCGA-3C-AALJ,-2.589002,0.206209,-0.121090,-3.404418,-0.407579,-2.548499,1.612541,0.648124,1.346423,1.277537,...,0.282041,-3.187522,-0.186341,-1.629101,-0.477434,0.001422,2.064204,-0.063295,0.622520,-2.277574
TCGA-3C-AALK,-2.702234,0.189963,-0.478374,-3.640364,-0.765470,-2.001555,1.773369,0.035087,1.397632,1.496949,...,0.332135,-3.114353,-0.352242,-1.645405,-0.359979,-0.045064,2.232906,-0.239828,0.523925,-2.081761
TCGA-4H-AAAK,-2.695440,0.289530,-0.275142,-3.467757,-0.668077,-2.160819,1.712105,-0.029550,1.124759,1.618586,...,0.547361,-3.377281,-0.202760,-1.718365,-0.147447,0.057300,2.076375,-0.726954,0.639236,-2.119115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-WT-AB44,-3.447477,1.103121,0.091124,-3.194960,-0.647661,-2.033286,1.549412,0.429877,1.238971,1.400613,...,0.824369,-3.029050,-0.336251,-1.752728,-0.587702,-0.159741,1.263707,-0.580195,-0.343583,-1.466069
TCGA-XX-A899,-2.575941,0.290430,-0.584301,-3.704024,-0.856065,-2.290457,1.792168,0.474566,1.410785,0.943516,...,0.439283,-3.303411,-0.091431,-1.603314,-0.420519,0.241184,2.095715,-0.239367,0.454681,-2.458776
TCGA-XX-A89A,-3.200147,0.073549,-0.345913,-3.451575,-0.874071,-2.487426,1.833855,0.194701,1.787738,1.114411,...,0.648099,-2.636375,-0.032807,-2.029949,-0.267664,-0.379086,1.926854,-0.068750,0.915008,-2.083760


In [12]:
df_eva_emb

,eva_0,eva_1,eva_2,eva_3,eva_4,eva_5,eva_6,eva_7,eva_8,eva_9,...,eva_246,eva_247,eva_248,eva_249,eva_250,eva_251,eva_252,eva_253,eva_254,eva_255
patient_id,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU,-2.155622,-0.291868,0.169682,-3.533919,-0.281709,-2.095984,1.502106,1.295753,0.824450,1.147492,...,0.673566,-2.987129,0.246001,-1.612530,-0.914301,-0.229371,1.868367,-0.552698,0.517181,-2.520375
TCGA-3C-AALI,-2.378538,-0.091101,0.088475,-3.407782,-0.398164,-2.097007,1.655441,0.476619,1.194781,1.192048,...,0.011487,-2.996671,-0.277262,-1.663273,-0.839044,-0.232655,1.803326,0.155092,0.614532,-2.153844
TCGA-3C-AALJ,-2.589002,0.206209,-0.121090,-3.404418,-0.407579,-2.548499,1.612541,0.648124,1.346423,1.277537,...,0.282041,-3.187522,-0.186341,-1.629101,-0.477434,0.001422,2.064204,-0.063295,0.622520,-2.277574
TCGA-3C-AALK,-2.702234,0.189963,-0.478374,-3.640364,-0.765470,-2.001555,1.773369,0.035087,1.397632,1.496949,...,0.332135,-3.114353,-0.352242,-1.645405,-0.359979,-0.045064,2.232906,-0.239828,0.523925,-2.081761
TCGA-4H-AAAK,-2.695440,0.289530,-0.275142,-3.467757,-0.668077,-2.160819,1.712105,-0.029550,1.124759,1.618586,...,0.547361,-3.377281,-0.202760,-1.718365,-0.147447,0.057300,2.076375,-0.726954,0.639236,-2.119115
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-WT-AB44,-3.447477,1.103121,0.091124,-3.194960,-0.647661,-2.033286,1.549412,0.429877,1.238971,1.400613,...,0.824369,-3.029050,-0.336251,-1.752728,-0.587702,-0.159741,1.263707,-0.580195,-0.343583,-1.466069
TCGA-XX-A899,-2.575941,0.290430,-0.584301,-3.704024,-0.856065,-2.290457,1.792168,0.474566,1.410785,0.943516,...,0.439283,-3.303411,-0.091431,-1.603314,-0.420519,0.241184,2.095715,-0.239367,0.454681,-2.458776
TCGA-XX-A89A,-3.200147,0.073549,-0.345913,-3.451575,-0.874071,-2.487426,1.833855,0.194701,1.787738,1.114411,...,0.648099,-2.636375,-0.032807,-2.029949,-0.267664,-0.379086,1.926854,-0.068750,0.915008,-2.083760


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# df_eva_emb.to_parquet('/content/drive/MyDrive/Datasets/fydp/rna_eva_embeddings.parquet')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
